# Fine-tuning Llama 3.2 1B para el NLG de VALERIA
Este notebook:
1. Instala dependencias
2. Descarga Llama 3.2 1B desde Hugging Face (requiere token + acceso aprobado)
3. Clona tu repo `Valeria-6.0` y carga el dataset de `ENTRENAMIENTO/`
4. Hace fine-tuning con LoRA (rápido y liviano, apto para GPU gratuita de Kaggle)
5. Prueba el modelo resultante y lo guarda

**Antes de correr:** en Kaggle, andá a Settings (panel derecho) → Accelerator → elegí **GPU T4 x2** (o P100). Sin esto, todo corre en CPU y es MUY lento.


## 1. Instalar dependencias

In [ ]:
!pip install -q -U transformers accelerate peft bitsandbytes datasets trl


## 2. Login a Hugging Face
Llama 3.2 es un modelo "gated": necesitás:
1. Una cuenta en https://huggingface.co
2. Pedir acceso en https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct (aprobación automática, tarda minutos)
3. Un token en https://huggingface.co/settings/tokens (tipo "Read")
4. En Kaggle: panel derecho → Add-ons → Secrets → agregá un secret llamado `HF_TOKEN` con ese valor (así no lo escribís a mano acá)


In [ ]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)


## 3. Clonar tu repo y cargar el dataset
Reemplazá la URL si tu repo es privado vas a necesitar un token de GitHub también (mismo mecanismo de Secrets, agregando `GH_TOKEN`).


In [ ]:
import subprocess

REPO_URL = "https://github.com/Mauricio7762/Valeria-6.0.git"
subprocess.run(["git", "clone", REPO_URL, "valeria_repo"], check=True)


In [ ]:
import json

DATASET_PATH = "valeria_repo/ENTRENAMIENTO/dataset_valeria_finetune.jsonl"

ejemplos = []
with open(DATASET_PATH, encoding="utf-8") as f:
    for linea in f:
        ejemplos.append(json.loads(linea))

print(f"Cargados {len(ejemplos)} ejemplos")
print(ejemplos[0])


## 4. Formatear el dataset al formato de chat de Llama 3.2

In [ ]:
from datasets import Dataset

SYSTEM_PROMPT = (
    "Sos el módulo de lenguaje natural de VALERIA, un sistema de IA. "
    "Respondé de forma clara, natural y breve, en español."
)

def formatear(ej):
    texto = (
        f"<|start_header_id|>system<|end_header_id|>\n\n{SYSTEM_PROMPT}<|eot_id|>"
        f"<|start_header_id|>user<|end_header_id|>\n\n{ej['instruction']}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>\n\n{ej['output']}<|eot_id|>"
    )
    return {"text": texto}

dataset = Dataset.from_list(ejemplos).map(formatear)
dataset = dataset.train_test_split(test_size=0.1, seed=7)
print(dataset)
print(dataset["train"][0]["text"])


## 5. Cargar el modelo base en 4-bit (para que entre en la GPU gratuita)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)


## 6. Configurar LoRA

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


## 7. Entrenar

In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="valeria_llama_lora",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=5,
    save_strategy="epoch",
    eval_strategy="epoch",
    bf16=True,
    dataset_text_field="text",
    max_seq_length=512,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
)

trainer.train()


## 8. Probar el modelo entrenado

In [ ]:
def preguntar(pregunta):
    prompt = (
        f"<|start_header_id|>system<|end_header_id|>\n\n{SYSTEM_PROMPT}<|eot_id|>"
        f"<|start_header_id|>user<|end_header_id|>\n\n{pregunta}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>\n\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    salida = model.generate(**inputs, max_new_tokens=100, temperature=0.7, do_sample=True)
    return tokenizer.decode(salida[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print(preguntar("¿Qué es el sistema glial?"))
print(preguntar("¿Cuál es la función de la microglía?"))


## 9. Guardar el resultado
Esto guarda solo los adaptadores LoRA (livianos, unos MB). Para usarlos después, cargás el modelo base + estos adaptadores con `PeftModel.from_pretrained`.


In [ ]:
model.save_pretrained("valeria_llama_lora_final")
tokenizer.save_pretrained("valeria_llama_lora_final")
print("Guardado. Lo encontrás en el panel de Output de Kaggle para descargar.")
